# KAN-RFB Tutorial — import, data, visualize, train, test

Demonstrates the full pipeline: loading data, plotting bubble shapes,
training both modes (constant & xi) via `train_multi_bubble_on_dataset`,
and testing in the static-condensation assembly.

---
## Colab setup
Run this cell first if using Google Colab (otherwise skip it).

In [ ]:
# ── 0. COLAB SETUP (run first if using Colab) ─────────────────────────────
# Clone repo from GitHub (no tarball upload needed)
!git clone https://github.com/juanpaca/ML_Galerkin_proj.git /content/ML_Galerkin_proj
%cd /content/ML_Galerkin_proj

import torch
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# ── 1. IMPORTS ──────────────────────────────────────────────────────────
import sys, os
# In Colab, CWD is /content; in local, it's the project root
if os.path.isdir('/content/ML_Galerkin_proj'):
    sys.path.insert(0, '/content/ML_Galerkin_proj')
else:
    sys.path.insert(0, '.')

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.dataset_generation import load_dataset, dataset_summary, train_multi_bubble_on_dataset
from src.rfb_bubble import KANBubble1D, MultiKANBubble1D
from src.rfb_exact import ExactRFBubbleSet1D
from src.rfb_assembly import (
    assemble_classical_system, assemble_rfb_condensed_system,
    recover_bubble_coefficients, RFBSolution1D,
)
from src.mesh import Mesh1D
from src.quadrature import GaussLegendre
from src.pde import AdvectionDiffusion1D
from src.basis import LagrangeBasis1D
from src.errors import compute_l2_error, relative_error_percentage

# Move training to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── 2. LOAD DATASET ─────────────────────────────────────────────────────
ds = load_dataset("rfb_1k")
dataset_summary(ds)

# Dataset structure:
#   ds["train"]["constant"]  →  {pe, rho, xi, b, db}  (797 samples)
#   ds["train"]["xi"]         →  {pe, rho, xi, b, db}
#   ds["val"][...]            →  validation split
#   ds["test"][...]           →  test split

xi_full = ds["train"]["constant"]["xi"]  # original FD grid
print(f"xi grid: {xi_full.shape[0]} points")

In [ ]:
# ── 3. VISUALIZE BUBBLE SHAPES IN THE DATASET ────────────────────────────
mode = "constant"
train_data = ds["train"][mode]
idx = [0, 100, 200, 300]

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, i in zip(axes.flat, idx):
    ax.plot(xi_full, train_data["b"][i])
    ax.set_title(f"Pe={train_data['pe'][i]:.2e}, ρ={train_data['rho'][i]:.2e}")
    ax.grid(True, alpha=0.3)
fig.supxlabel("ξ")
fig.supylabel("b̂(ξ)")
fig.tight_layout()
plt.show()

print("Left: low Pe (diffusion-dominated)  →  rounded, symmetric")
print("Right: high Pe (advection-dominated) →  sharp boundary layer at ξ=1")

In [ ]:
# ── 4. VISUALIZE THE CELL-BASED SPLIT ───────────────────────────────────
cell_map = ds["metadata"]["cell_map"]
pe_bins  = ds["metadata"].get("pe_bins", [0.3, 1, 10, 100, 1000, 30000])

fig, ax = plt.subplots(figsize=(8, 5))
colors = {"train": "#1F4E79", "val": "#F7DC6F", "test": "#E74C3C"}
markers = {"train": "o", "val": "s", "test": "^"}

for sname in ["train", "val", "test"]:
    pe_s = ds[sname]["constant"]["pe"]
    rho_s = ds[sname]["constant"]["rho"]
    ax.scatter(pe_s, rho_s, c=colors[sname], marker=markers[sname],
               alpha=0.4, s=10, label=sname, edgecolors='none')

ax.set_xscale("log")
ax.set_xlabel("Pe")
ax.set_ylabel("ρ")
ax.set_title("Cell-based split: train / val / test — no leakage")
ax.legend()
ax.grid(True, alpha=0.3)
for pb in pe_bins:
    ax.axvline(pb, color="gray", lw=0.5, ls="--")
plt.show()

train_cells = {k for k, (s, _) in cell_map.items() if s == "train"}
val_cells   = {k for k, (s, _) in cell_map.items() if s == "val"}
test_cells  = {k for k, (s, _) in cell_map.items() if s == "test"}
print(f"Cells: {len(train_cells)} train / {len(val_cells)} val / {len(test_cells)} test")
print(f"Overlap train∩val: {len(train_cells & val_cells)}")
print(f"Overlap train∩test: {len(train_cells & test_cells)}")

In [ ]:
# ── 5. TRAINING (both modes) ────────────────────────────────────────────
# Uses the built-in train_multi_bubble_on_dataset which trains each
# bubble independently with value-only loss (grad_weight=0).

multi_model = MultiKANBubble1D(n_bubbles=2, n_hidden=5, n_grid=8, spline_order=3)
n_params = sum(p.numel() for p in multi_model.parameters())
print(f"MultiKANBubble1D: {n_params} params ({n_params // 2} per mode)\n")

histories = train_multi_bubble_on_dataset(
    multi_model, ds["train"],
    mode_names=("constant", "xi"),
    n_epochs=300, batch_size=256, lr=1e-3,
    grad_weight=0.0,   # value-only — gradient term diverges
    n_quad=100,
    verbose=True,
    device=device,
)
print("\nTraining done!")

In [ ]:
# ── 5B. LOAD PRETRAINED MODEL (use if training interrupted) ────────────────
# Replace multi_model with the pre-trained one
multi_model = MultiKANBubble1D(n_bubbles=2, n_hidden=5, n_grid=8, spline_order=3)
multi_model.load_state_dict(torch.load('models/multi_bubble_model_1k.pt'))
print("Loaded pretrained model from models/multi_bubble_model_1k.pt")


In [ ]:
# ── 5C. LOSS CURVES (only if training was run) ─────────────────────────────
if 'histories' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for ax, (mname, hist) in zip(axes, histories.items()):
        ax.plot(hist)
        ax.set_yscale('log')
        ax.set_xlabel('epoch')
        ax.set_ylabel('MSE')
        ax.set_title(f"{mname} — final loss: {hist[-1]:.4e}")
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("Training not run — skipping loss curves.")

In [ ]:
# ── 5D. TEST BOTH MODES ON THE HELD-OUT TEST SET ────────────────────────
xi_full_t = torch.tensor(xi_full, dtype=torch.float32)
n_fd = len(xi_full)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (i, mname) in enumerate([(0, "constant"), (1, "xi")]):
    test_data = ds["test"][mname]
    p_te = torch.tensor(test_data["pe"], dtype=torch.float32)
    r_te = torch.tensor(test_data["rho"], dtype=torch.float32)
    n_te = p_te.shape[0]

    with torch.no_grad():
        xi_et = xi_full_t.unsqueeze(0).expand(n_te, -1).reshape(-1)
        p_et  = p_te.unsqueeze(1).expand(-1, n_fd).reshape(-1)
        r_et  = r_te.unsqueeze(1).expand(-1, n_fd).reshape(-1)
        pred = multi_model.bubbles[i](xi_et, p_et, r_et).reshape(n_te, n_fd)
        mse = F.mse_loss(pred, torch.tensor(test_data["b"], dtype=torch.float32)).item()
        rms = np.sqrt(np.mean((pred.numpy() - test_data["b"])**2, axis=1))

    axes[ax].scatter(test_data["pe"], rms, alpha=0.6, s=15)
    axes[ax].set_xscale('log')
    axes[ax].set_xlabel('Pe')
    axes[ax].set_ylabel('RMSE per sample')
    axes[ax].set_title(f"{mname}  —  MSE={mse:.4e}")
    axes[ax].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# ── 6. COMPARE BUBBLE SHAPES: KAN vs EXACT ──────────────────────────────
# Uses actual samples from the HELD-OUT test set (cell-based split).
# The model has never seen these (Pe, ρ) regimes during training.

xi_plot = np.linspace(0, 1, 201)
H = 1/16; BETA = 1.0

# Pick 4 samples from the test set, spread across the test Pe-ρ range
test_data_const = ds["test"]["constant"]
test_idx = [0, 50, 100, 140]
pe_samples = test_data_const["pe"][test_idx]
rho_samples = test_data_const["rho"][test_idx]

fig, axes = plt.subplots(4, 2, figsize=(10, 12), sharex=True, sharey='row')
for row, (pe, rho) in enumerate(zip(pe_samples, rho_samples)):
    eps = BETA * H / (2 * pe) if pe > 0 else 1.0
    sigma = rho * eps / H**2

    b_kan, _ = multi_model.value_grad_numpy(xi_plot, pe, rho)

    exact = ExactRFBubbleSet1D(eps, BETA, sigma, H,
                                residual_modes=("constant", "xi"), n_points=4000)
    b_ex, _ = exact.value_grad_numpy(xi_plot, pe, rho)

    for col, (mode_name, k, e) in enumerate([("b̂ (constant)", 0, 0), ("b̃ (xi)", 1, 1)]):
        axes[row, col].plot(xi_plot, b_kan[k], label='KAN')
        axes[row, col].plot(xi_plot, b_ex[e], '--', label='Exact FD')
        axes[row, col].set_title(f"{mode_name}  Pe={pe:.2f}, ρ={rho:.2f}  (test cell)")
        axes[row, col].legend(fontsize=8)
        axes[row, col].grid(True, alpha=0.3)

fig.supxlabel('ξ')
fig.tight_layout()
plt.show()


In [ ]:
# ── 7. ASSEMBLY: classical vs KAN-RFB vs exact RFB ──────────────────────
# Uses a PDE configuration from the HELD-OUT test set.
# The KAN model has never seen this (Pe, ρ) during training.

# Take the first test sample's parameters
pe_test = ds["test"]["constant"]["pe"][60]  # ≈ 1.5
rho_test = ds["test"]["constant"]["rho"][60]  # ≈ 0.5
H = 1/20; BETA = 1.0
EPS = BETA * H / (2 * pe_test)  # ε such that Pe matches test cell
SIGMA = rho_test * EPS / H**2
N_EL = 20

print(f"Test case: ε={EPS:.4f}, β={BETA}, σ={SIGMA:.4f}")
print(f"  Pe={BETA*H/(2*EPS):.2f}, ρ={SIGMA*H**2/EPS:.2f}  (from test cell)")

mesh = Mesh1D(0.0, 1.0, N_EL)
quad = GaussLegendre(16)
pde = AdvectionDiffusion1D(EPS, BETA, SIGMA)
pde.set_source_from_function(lambda x: np.ones_like(x))

# ── Ground truth (fine FD) ──
N_FD = 8000
dx_ref = 1.0 / (N_FD - 1)
x_ref = np.linspace(0, 1, N_FD)
A_ref = np.zeros((N_FD, N_FD))
rhs_ref = np.ones(N_FD)
for i in range(1, N_FD - 1):
    A_ref[i, i-1] = -EPS/dx_ref**2 - BETA/(2*dx_ref)
    A_ref[i, i]   =  2*EPS/dx_ref**2 + SIGMA
    A_ref[i, i+1] = -EPS/dx_ref**2 + BETA/(2*dx_ref)
A_ref[0,0] = A_ref[-1,-1] = 1.0
rhs_ref[0] = rhs_ref[-1] = 0.0
u_exact = np.linalg.solve(A_ref, rhs_ref)

def exact_u(x): return np.interp(x, x_ref, u_exact)

# ── 1. Classical P1 ──
A_cl, F_cl = assemble_classical_system(mesh, quad, pde)
u_cl = np.linalg.solve(A_cl, F_cl)

class ClassicalSolution:
    def __init__(self, mesh, u):
        self.mesh = mesh; self.u = u
    def __call__(self, x):
        basis = LagrangeBasis1D(self.mesh)
        out = np.zeros_like(x)
        for i in range(self.mesh.n_nodes):
            out += self.u[i] * basis.eval(x, i)
        return out

l2_cl, norm = compute_l2_error(ClassicalSolution(mesh, u_cl), exact_u)
print(f"\nClassical P1:  L2 = {l2_cl:.4e}  ({relative_error_percentage(l2_cl, norm):.1f}%)")

# ── 2. Exact RFB ──
exact_set = ExactRFBubbleSet1D(EPS, BETA, SIGMA, mesh.h,
                                residual_modes=("constant", "xi"), n_points=8000)
A_ex, F_ex, local_ex = assemble_rfb_condensed_system(mesh, quad, pde, exact_set)
u_ex = np.linalg.solve(A_ex, F_ex)
ub_ex = recover_bubble_coefficients(u_ex, mesh, local_ex)
sol_ex = RFBSolution1D(u_ex, ub_ex, mesh, exact_set, pde)
l2_ex, _ = compute_l2_error(sol_ex, exact_u)
print(f"Exact RFB:     L2 = {l2_ex:.4e}  ({relative_error_percentage(l2_ex, norm):.1f}%)")

# ── 3. KAN-RFB (trained both modes) ──
A_kan, F_kan, local_kan = assemble_rfb_condensed_system(mesh, quad, pde, multi_model)
u_kan = np.linalg.solve(A_kan, F_kan)
ub_kan = recover_bubble_coefficients(u_kan, mesh, local_kan)
sol_kan = RFBSolution1D(u_kan, ub_kan, mesh, multi_model, pde)
l2_kan, _ = compute_l2_error(sol_kan, exact_u)
print(f"KAN-RFB:       L2 = {l2_kan:.4e}  ({relative_error_percentage(l2_kan, norm):.1f}%)")

print(f"\n{'Method':<20} {'L2 error':<12} {'Rel %':<8}")
print(f"{'-'*20} {'-'*12} {'-'*8}")
for name, err in [("Classical P1", l2_cl), ("Exact RFB", l2_ex), ("KAN-RFB", l2_kan)]:
    print(f"{name:<20} {err:<12.4e} {relative_error_percentage(err, norm):<8.1f}")


In [ ]:
# ── 8. PLOT SOLUTIONS ───────────────────────────────────────────────────
x_p = np.linspace(0, 1, 200)
plt.figure(figsize=(9, 5))
plt.plot(x_ref, u_exact, 'k-', lw=1.5, label='Exact (FD ref)')
plt.plot(x_p, ClassicalSolution(mesh, u_cl)(x_p), '--', label=f'Classical P1')
plt.plot(x_p, sol_ex(x_p), ':', lw=2, label=f'Exact RFB (L2={l2_ex:.2e})')
plt.plot(x_p, sol_kan(x_p), '-.', lw=2, label=f'KAN-RFB (L2={l2_kan:.2e})')
plt.xlabel('x')
plt.ylabel('u(x)')
plt.legend()
plt.title(f'ε={EPS}, β={BETA}, σ={SIGMA} — Pe={BETA*mesh.h/(2*EPS):.1f}')
plt.grid(True, alpha=0.3)
plt.show()

## Next steps

- Save the trained model: `torch.save(multi_model.state_dict(), "models/multi_kan.pt")`
- Sweep over multiple ε values to see how enrichment behaves
- Try with reaction (σ > 0)
- Generate a larger dataset and retrain